# 7.3 · 交易接口与模拟盘

> 策略最终要通过**交易接口（网关）**把订单送到券商/交易所。交易接口隔离了"策略逻辑"和"具体柜台"，让同一套策略可以对接不同通道。
> 本节了解常见交易通道、统一接口设计、仿真环境，以及从开户到上线的完整对接流程。

## 一、交易接口的作用

- 向上提供统一的"下单/撤单/查持仓/查资金/收回报"接口；
- 向下对接具体柜台（券商、期货、交易所），处理协议、连接、认证；
- 屏蔽不同柜台的差异，策略代码不随通道改变；
- 处理断线重连、订单状态、成交回报、对账。

> 这是典型的"网关/适配器"设计模式，是量化开发岗位的常见工作。

## 二、常见交易通道

**股票（A股）：**
- 券商量化客户端：**QMT（迅投 miniQMT）、Ptrade（恒生）**，支持 Python，是个人/小机构最常用；
- 券商 API/柜台：恒生 UFT、金证、顶点；华泰 MATIC、国泰君安、中信等有机构接口；
- 交易所直连：仅大型机构（需席位、专线）。

**期货/期权：**
- **CTP（上期综合交易平台，上期技术）**：期货行业标准，C++ 接口，Python 有 openctp/vnpy 等封装；
- 仿真：CTP 官方 **SimNow** 模拟环境。

**其他：**
- 港股/美股：Interactive Brokers（IB）、富途、老虎、长桥等 OpenAPI。

## 三、统一接口抽象（代码在 live_lab.broker）

In [1]:
import sys
sys.path.insert(0, r'C:\Users\gucao\Desktop\Qlib')
from live_lab.broker import SimBroker
from live_lab.orders import Order
# 策略只依赖 BaseBroker 的统一方法，底层可换成真实券商/CTP
b = SimBroker(cash=1_000_000); b.connect()
o1 = Order('A', 10000, 'BUY', 'LIMIT', price=10.0)
b.place_order(o1, fill_price=10.0)
o2 = Order('A', 3000, 'SELL', 'LIMIT', price=11.0)
b.place_order(o2, fill_price=11.0)
print('持仓:', b.query_position())
print('现金: %.2f' % b.query_account()['cash'])

持仓: {'A': 7000}
现金: 933000.00


## 四、仿真与模拟盘

- **模拟盘（paper trading）**：用实时行情、真实系统流程，不接真钱（7.1 的 PaperBroker）；
- **CTP SimNow**：期货官方仿真，对接真实 CTP 协议、模拟撮合，用于验证 CTP 接口；
- 券商 QMT/Ptrade 也提供模拟环境。

> 仿真能验证接口对接、下单回报、系统流程，是实盘前的关键一步。

## 五、对接与上线流程

1. **开户/申请**：选择支持量化接口的券商，开通 QMT/Ptrade 或机构 API；期货开通 CTP；
2. **获取接口文档和账号**：了解字段、限制、费率、交易时间；
3. **开发对接**：实现 BaseBroker 具体子类，处理连接、认证、回报；
4. **仿真测试**：在模拟环境验证下单、撤单、持仓、资金、断线重连；
5. **小资金灰度**：用少量真实资金验证，与券商对账；
6. **逐步放量**：确认无误后增加资金，并持续监控。

## 六、对接注意事项

- **断线重连**：网络/柜台会断开，需自动重连并恢复订单状态；
- **成交回报**：订单可能部分成交、拒单、延迟，要异步处理回报；
- **对账**：定期与券商核对资金、持仓、成交，发现不一致及时处理；
- **流控/限频**：柜台对下单频率有限制，不能超频；
- **交易时间**：严格遵守开收盘、集合竞价、夜盘时间；
- **风控前置**：本地风控 + 柜台风控双重把关。

## 七、小结与面试要点

- 交易接口隔离策略与柜台：统一方法（下单/撤单/查询/回报），底层可换。
- 股票常用 QMT/Ptrade，期货用 CTP（SimNow 仿真），海外用 IB 等。
- 上线流程：开户→对接→仿真→小资金→放量；注意重连、回报、对账、流控。

**面试高频：**
- 策略怎么对接券商/交易所？（答：通过交易网关/接口，统一抽象底层柜台）
- 你了解哪些量化交易通道？（答：QMT、Ptrade、CTP、IB 等）
- 什么是 CTP / SimNow？（答：期货标准交易接口；其官方仿真环境）
- 实盘接口要处理哪些工程问题？（答：断线重连、成交回报、对账、流控、交易时间）

## 动手练习
1. 阅读 QMT 或 CTP 的接口文档，列出核心方法；
2. 实现一个对接 SimBroker 的简单策略；
3. 写出你从现在到真实下单的完整 checklist。

---
下一节 **7.4 实时风控与监控：事前/事中/事后风控、报警**。